# Join datasets
- Author: Bryan Bravo
- Created: 2026-03-28
## Import Libraries

In [1]:
import os
import sys

os.chdir("C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch")
# Set JAVA_HOME before importing PySpark and use findspark
os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-22'  # May need to remove or update in cloud environment.
import findspark
findspark.init()

import requests
import pandas as pd
import numpy as np
import json
import pyspark
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from functools import reduce
from pyspark.sql import (
    functions as F,
    Window as W,
    types as T,
    SparkSession,
    DataFrame
)

# Visualize data for joined dataframes.
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns


# api keys and other hardcoded values for the Strait of Hormuz Research project.
# Note: In a production environment, these should be stored securely and not hardcoded.
import hardcoded_keys # Note: This file is added to .gitignore to prevent accidental commits of sensitive information.

import proj_vars

### Initialize Spark Session


In [2]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BusinessPlanAnalysis") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.parquet.nativeio.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")

Spark Session created successfully!


## Variables

In [3]:
end_date = (dt.now().replace(day=1) - relativedelta(days=1)).strftime("%Y-%m-%d")
# in_path = 's3a://ml-project-s3-bronze/input_folder/'
in_path = 'processed_datasets/'
out_path = 'processed_datasets/'


## Custom Functions

In [4]:
def detect_outliers_by_partition(
    df,
    numeric_cols,
    partition_cols=['country', 'year'],
    date_col='date',
    date_format='yyyyMMdd'
):
    """
    Detect outliers using the IQR method partitioned by specified columns.
    
    Uses vectorized window functions for efficient PySpark processing with
    temporal and geopolitical context awareness through partitioning.
    
    Parameters:
    -----------
    df : pyspark.sql.DataFrame
        Input DataFrame containing the data to analyze
    numeric_cols : list
        List of numeric column names to check for outliers
    partition_cols : list, default=['country', 'year']
        Columns to partition by for outlier detection
    date_col : str, default='date'
        Name of the date column (used to extract year if needed)
    date_format : str, default='yyyyMMdd'
        Format string for parsing the date column
    
    Returns:
    --------
    tuple : (original_df, outliers_df, outlier_summary)
        - original_df: Input DataFrame with year column added (if applicable)
        - outliers_df: DataFrame of all detected outliers across columns
        - outlier_summary: Dictionary with outlier counts per numeric column
    
    Example:
    --------
    >>> clean_df, outliers_df, summary = detect_outliers_by_partition(
    ...     joined_df1, 
    ...     ['interest_rate', 'fx_rate', 'brent_dollars_per_barrel']
    ... )
    """
    
    # Add year column if partitioning by year and it doesn't exist
    if 'year' in partition_cols and 'year' not in df.columns:
        df = df.withColumn('year', F.year(F.to_date(date_col, date_format)))
    
    # Define window specification
    w = W.partitionBy(*partition_cols)
    
    outlier_summary = {}
    outliers_list = []
    row_count = df.count()
    
    for col in numeric_cols:
        print(f"\nOutliers for column: {col}")
        
        # Vectorized outlier detection via window functions
        df_flagged = (
            df
            .withColumn('q1', F.percentile_approx(col, 0.25).over(w))
            .withColumn('q3', F.percentile_approx(col, 0.75).over(w))
            .withColumn('iqr', F.col('q3') - F.col('q1'))
            .withColumn(
                'is_outlier',
                (F.col(col) < (F.col('q1') - 1.5 * F.col('iqr'))) |
                (F.col(col) > (F.col('q3') + 1.5 * F.col('iqr')))
            )
            .filter(F.col('is_outlier'))
            .withColumn('outlier_column', F.lit(col))
            .withColumn('outlier_value', F.col(col))
            .select('country', 'date', 'year', 'outlier_column', 'outlier_value', 'q1', 'q3', 'iqr')
        )
        
        # Collect outliers from this column
        outliers_list.append(df_flagged)
        
        # Aggregate outliers per country
        outlier_counts = (
            df_flagged
            .groupBy('country')
            .count()
            .orderBy(F.desc('count'))
        )
        
        outlier_counts.show()
        
        # Get total count
        total = outlier_counts.agg(F.sum('count')).collect()[0][0] or 0
        outlier_summary[col] = total
        print(f"Total outliers in column '{col}': {total} out of {row_count} rows ({(total / row_count) * 100:.2f}%)")
    
    # Combine all outliers into a single dataframe
    if outliers_list:
        outliers_df = outliers_list[0]
        for outliers_chunk in outliers_list[1:]:
            outliers_df = outliers_df.union(outliers_chunk)
        
        print("\n--- --- --- --- --- --- --- --- ---\nCombined Outliers DataFrame:")
        print(f"Total outlier records: {outliers_df.count()}")
        outliers_df.show(10)
    else:
        outliers_df = None
    
    print("\n--- --- --- --- --- --- --- --- ---\nSummary:")
    for col, total in outlier_summary.items():
        print(f"Column '{col}': {total} total outliers ({(total / row_count) * 100:.2f}%)")
    
    return df, outliers_df, outlier_summary


In [5]:
# Read and display the first few rows of each dataset to verify they are loaded correctly.
df_list = ['acled', 'cpi', 'fred', 'gpr', 'lsci', 'oil', 'wb']
for df_name in df_list:
    globals()[f"{df_name}_df"] = spark.read.csv(f"{in_path}/{df_name}.csv", header=True, inferSchema=True)
    print(f"DataFrame: {df_name}")
    globals()[f"{df_name}_df"].show(5)


DataFrame: acled
+---------+-----+----+------+
|  country|month|year|events|
+---------+-----+----+------+
|australia|    1|2021|     2|
|australia|    2|2021|     0|
|australia|    3|2021|     0|
|australia|    4|2021|     0|
|australia|    5|2021|     1|
+---------+-----+----+------+
only showing top 5 rows

DataFrame: cpi
+---------+-----+----+-----+
|  country|value|year|month|
+---------+-----+----+-----+
|australia| 96.4|2024|    4|
|australia|96.17|2024|    5|
|australia|96.52|2024|    6|
|australia|96.77|2024|    7|
|australia|96.49|2024|    8|
+---------+-----+----+-----+
only showing top 5 rows

DataFrame: fred
+--------+---------+-------------+-------+
|    date|  country|interest_rate|fx_rate|
+--------+---------+-------------+-------+
|20060403|australia|          5.5| 0.7177|
|20060404|australia|          5.5| 0.7203|
|20060405|australia|          5.5| 0.7263|
|20060406|australia|          5.5| 0.7315|
|20060407|australia|          5.5| 0.7279|
+--------+---------+-------

## Join Datasets and Cleaning
### FRED and Oil datasets
Joining and checking for duplicates

In [6]:
# FRED and oil
joined_df1 = fred_df.join(oil_df, on='date', how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df1.count(), len(joined_df1.columns))}")
print("Columns in joined_df1: ", joined_df1.columns)

## Check for duplicates
duplicate_count = joined_df1.groupBy(joined_df1.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df1: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 6)
Columns in joined_df1:  ['date', 'country', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df1: 0


No duplicate rows were found.
checking for missing values

In [7]:
## Check for missing values
missing_values = {col: joined_df1.filter(joined_df1[col].isNull()).count() for col in joined_df1.columns}
print("Missing values in joined_df1:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df1:
--- --- --- --- --- --- --- --- ---
date: 0
country: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 1278
wti_dollars_per_barrel: 1278


#### Missing data treatment
A close inspection of the dates in relation to the missing oil-price data reveals that they fall on **market holidays or non-trading days**. Oil benchmarks do not publish prices on these days. This is expected behavior.
- Using F-Fill on the missing data is applicable in this situation based on the assumption that the _true price_ is the last traded price for this analysis.

In [8]:
(joined_df1.filter(F.col('brent_dollars_per_barrel').isNull()
                   | F.col('wti_dollars_per_barrel').isNull())
.orderBy('country', 'date').show())

+--------+---------+----------------+-------+------------------------+----------------------+
|    date|  country|   interest_rate|fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|
+--------+---------+----------------+-------+------------------------+----------------------+
|20060414|australia|             5.5| 0.7289|                    NULL|                  NULL|
|20060417|australia|             5.5| 0.7375|                    NULL|                  NULL|
|20060703|australia|            5.75| 0.7436|                    NULL|                  NULL|
|20061029|australia|             6.0| 0.7692|                    NULL|                  NULL|
|20061124|australia|6.19318181818182| 0.7782|                    NULL|                  NULL|
|20061226|australia|            6.25| 0.7829|                    NULL|                  NULL|
|20070406|australia|            6.25| 0.8177|                    NULL|                  NULL|
|20070409|australia|            6.25| 0.8167|               

In [9]:
# F-Fill missing values for oil prices in joined_df1.
w1 = W.partitionBy('country').orderBy('date')
joined_df1 = (
    joined_df1
    .withColumns({
        col: F.last(F.col(col), ignorenulls=True).over(w1) for col in ['brent_dollars_per_barrel', 'wti_dollars_per_barrel']
    })
)

## Check for missing values
missing_values = {col: joined_df1.filter(joined_df1[col].isNull()).count() for col in joined_df1.columns}
print("Missing values in joined_df1:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")

Missing values in joined_df1:
--- --- --- --- --- --- --- --- ---
date: 0
country: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0


#### Check for outliers
Due to how dynamic the data is for a 10 year period, the outliers will be checked per country on a yearly basis to capture temporal trends and ensure more accurate outlier detection.
- Notes:
    - The outliers found in these variables are likely not outliers but instead regime shifts. I'm looking for moments where geopolitical shocks, monitary policy changes, liquidity colapses, and capital controls occured. With this assumption, the IQR outlier is most likely a "signal" that the global economy experienced a shock.

In [10]:
numeric_cols = ['interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel']
joined_df1, outliers_df, outlier_summary = detect_outliers_by_partition(
    joined_df1,
    numeric_cols,
    partition_cols=['country', 'year'],
    date_col='date'
)



Outliers for column: interest_rate
+--------------+-----+
|       country|count|
+--------------+-----+
|        canada|  448|
|         japan|  429|
| united_states|  386|
|         india|  368|
|   south_korea|  326|
|united_kingdom|  308|
|        brazil|  285|
|        france|  266|
|     australia|  240|
|  south_africa|  207|
|        mexico|  103|
|         china|  102|
|       germany|    9|
|         italy|    9|
|        russia|    7|
|       turkiye|    6|
+--------------+-----+

Total outliers in column 'interest_rate': 3499 out of 60520 rows (5.78%)

Outliers for column: fx_rate
+--------------+-----+
|       country|count|
+--------------+-----+
|        mexico|  132|
|         japan|   96|
|        canada|   90|
|        brazil|   83|
|  south_africa|   75|
|   south_korea|   72|
| united_states|   49|
|         india|   38|
|united_kingdom|   35|
|     australia|   28|
|        france|   26|
|         china|   21|
|        russia|    7|
|       turkiye|    5|
|       g

Upon Closer inspection of the years for each variable we discover that each outliers fall within a geopolitical event such as the 2008 recession, COVID 2020 for rise in fuel costs. 

In [11]:
for col in numeric_cols:
    print(f"\nOutlier distribution over years for column: {col}")
    (outliers_df.filter(f"outlier_column == '{col}'")
     .groupBy('year').count().orderBy(F.desc('count')).show())


Outlier distribution over years for column: interest_rate
+----+-----+
|year|count|
+----+-----+
|2016|  436|
|2008|  433|
|2020|  390|
|2014|  344|
|2024|  207|
|2009|  204|
|2025|  188|
|2010|  182|
|2012|  167|
|2015|  146|
|2018|  120|
|2013|  105|
|2011|  104|
|2019|  103|
|2017|  102|
|2023|   99|
|2021|   83|
|2006|   42|
|2007|   41|
|2022|    3|
+----+-----+


Outlier distribution over years for column: fx_rate
+----+-----+
|year|count|
+----+-----+
|2008|  195|
|2006|   83|
|2017|   73|
|2014|   62|
|2020|   56|
|2024|   50|
|2021|   47|
|2012|   41|
|2016|   35|
|2023|   33|
|2022|   18|
|2009|   13|
|2013|   13|
|2007|   10|
|2018|   10|
|2019|    8|
|2025|    7|
|2015|    5|
|2011|    2|
+----+-----+


Outlier distribution over years for column: brent_dollars_per_barrel
+----+-----+
|year|count|
+----+-----+
|2020|  380|
|2014|  272|
|2008|  198|
|2012|  140|
|2018|   73|
|2011|   64|
|2017|   28|
|2021|   12|
|2006|    4|
|2016|    1|
+----+-----+


Outlier distribution 

### Joined and ACLED dataset

In [12]:
# joined and acled
joined_df1 = (
    joined_df1.withColumns({
        'year': F.year(F.to_date('date', 'yyyyMMdd')),
        'month': F.month(F.to_date('date', 'yyyyMMdd'))
    })
)

joined_df2 = joined_df1.join(acled_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df2.count(), len(joined_df2.columns))}")
print("Columns in joined_df2: ", joined_df2.columns)


## Check for duplicates
duplicate_count = joined_df2.groupBy(joined_df2.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df2: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 9)
Columns in joined_df2:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df2: 0


In [13]:
## Check for missing values
missing_values = {col: joined_df2.filter(joined_df2[col].isNull()).count() for col in joined_df2.columns}
print("Missing values in joined_df2:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df2:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 39091


Missing Data from the `events` variable is expected. The missing variable is indicative that ACLED was not tracking or didn't need to record violent conflict data.
#### Missing Data treatment
- Missing data from the `events` variable will be assumed that `0` conflicts occured, all missing values will be filled in as 0.

In [14]:
# fill missing values in the events column with 0 (assuming missing means no events)
joined_df2 = joined_df2.withColumn('events', F.when(F.col('events').isNull(), 0).otherwise(F.col('events')))

## Check for missing values
missing_values = {col: joined_df2.filter(joined_df2[col].isNull()).count() for col in joined_df2.columns}
print("Missing values in joined_df2:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df2:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0


#### Outlier detection on `events` column
Most variables in the `events` column is `0`, so any value that is flagged as an outlier is a tracked geopolitical event.

In [15]:
# Outlier detection for the 'events' column in joined_df2
numeric_cols = ['events']
joined_df2, outliers_df, outlier_summary = detect_outliers_by_partition(
    joined_df2,
    numeric_cols,
    partition_cols=['country', 'year'],
    date_col='date'
)


Outliers for column: events
+--------------+-----+
|       country|count|
+--------------+-----+
|   south_korea|  266|
|         china|  147|
|         india|  129|
|  south_africa|  104|
|     australia|   86|
|         japan|   83|
|        france|   83|
|        canada|   63|
|        mexico|   60|
|        brazil|   41|
| united_states|   41|
|united_kingdom|   22|
|         italy|    4|
|       germany|    2|
+--------------+-----+

Total outliers in column 'events': 1131 out of 60520 rows (1.87%)

--- --- --- --- --- --- --- --- ---
Combined Outliers DataFrame:
Total outlier records: 1131
+---------+--------+----+--------------+-------------+---+---+---+
|  country|    date|year|outlier_column|outlier_value| q1| q3|iqr|
+---------+--------+----+--------------+-------------+---+---+---+
|australia|20231002|2023|        events|            1|  0|  0|  0|
|australia|20231003|2023|        events|            1|  0|  0|  0|
|australia|20231004|2023|        events|            1|  0|  0

The `acled_df` has `monthly` occurences and is being joined by a dataframe which is `daily` based.
- To address this, the `events` column will only include a value on the last day of each month per country.
- The last day of the month is to take into account how a political event usually occurs after a global shock is introduced. All other values in the `events` column will be assumed as `0`.

In [16]:
# last.events
joined_df2 = (
    joined_df2
    .withColumn('last_events',
                F.row_number().over(W.partitionBy('country', 'year', 'month').orderBy(F.desc('date'))))
    .withColumn('events',
                F.when(F.col('last_events') == 1, F.col('events')).otherwise(0))
    .drop('last_events')
)

In [17]:
# Outlier detection for the 'events' column in joined_df2
numeric_cols = ['events']
joined_df2, outliers_df, outlier_summary = detect_outliers_by_partition(
    joined_df2,
    numeric_cols,
    partition_cols=['country', 'year'],
    date_col='date'
)


Outliers for column: events
+--------------+-----+
|       country|count|
+--------------+-----+
|  south_africa|  235|
|         india|  122|
|        brazil|   98|
|        mexico|   98|
|         china|   90|
| united_states|   74|
|        france|   73|
|united_kingdom|   61|
|        canada|   32|
|   south_korea|   26|
|     australia|   19|
|         japan|    4|
|         italy|    4|
|       germany|    2|
+--------------+-----+

Total outliers in column 'events': 938 out of 60520 rows (1.55%)

--- --- --- --- --- --- --- --- ---
Combined Outliers DataFrame:
Total outlier records: 938
+---------+--------+----+--------------+-------------+---+---+---+
|  country|    date|year|outlier_column|outlier_value| q1| q3|iqr|
+---------+--------+----+--------------+-------------+---+---+---+
|australia|20210129|2021|        events|            2|  0|  0|  0|
|australia|20210528|2021|        events|            1|  0|  0|  0|
|australia|20210730|2021|        events|            2|  0|  0| 

In [18]:
joined_df2.filter((F.col('year')==2023)&(F.col('month')==10)).show(10)

+----+-----+---------+--------+-------------+-------+------------------------+----------------------+------+
|year|month|  country|    date|interest_rate|fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|events|
+----+-----+---------+--------+-------------+-------+------------------------+----------------------+------+
|2023|   10|australia|20231031|          4.1| 0.6332|                   86.82|                 81.64|     1|
|2023|   10|australia|20231030|          4.1| 0.6374|                   90.73|                 83.03|     0|
|2023|   10|australia|20231027|          4.1| 0.6352|                   90.73|                 86.04|     0|
|2023|   10|australia|20231026|          4.1| 0.6312|                   88.45|                  83.8|     0|
|2023|   10|australia|20231025|          4.1| 0.6334|                   90.14|                 86.07|     0|
|2023|   10|australia|20231024|          4.1| 0.6349|                    88.0|                 84.58|     0|
|2023|   10|austral

### Joined and cpi dataset

In [19]:
# joined and cpi
cpi_df = cpi_df.withColumnRenamed('value', 'cpi')

joined_df3 = joined_df2.join(cpi_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df3.count(), len(joined_df3.columns))}")
print("Columns in joined_df3: ", joined_df3.columns)


## Check for duplicates
duplicate_count = joined_df3.groupBy(joined_df3.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df3: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 10)
Columns in joined_df3:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events', 'cpi']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df3: 0


No duplicates found
#### Checking for Missing Values

In [20]:
## Check for missing values
missing_values = {col: joined_df3.filter(joined_df3[col].isNull()).count() for col in joined_df3.columns}
print("Missing values in joined_df3:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df3:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 4609


# Missing data treatment
Missing values stems from the lack of data for `australia`. The data gathered from SDMX IMF API begins at 2024-04 for `australia`. Additionally, there are some countries that is lacking data for the year 2026.
- To correct the missing Australian data, cpi data will be imported from the Austrialian Bureau of Statistics.
    - Data from "https://www.abs.gov.au/statistics/economy/price-indexes-and-inflation/consumer-price-index-australia/sep-quarter-2025#data-downloads"
    - The downloaded CPI data is reported quarterly. F-Fill will be used to fill in missing values for months when the quarterly value is not reported, assuming the CPI indicator remains the same until the Australian Bureau of Statistics reports a new record.
- For the countries where the data is missing after 2025, the missing window of data is small (about 2 months). To address this data F-fill will be used.

In [21]:
(
    cpi_df
    .withColumn('date', F.concat(F.col('year'), F.lpad('month', 2, '0')))
    .groupBy('country').agg(
        F.min('year').alias('min_year'),
        F.max('year').alias('max_year'),
        F.min('date').alias('min_date'),
        F.max('date').alias('max_date')
    ).orderBy('country')
).show()

+--------------+--------+--------+--------+--------+
|       country|min_year|max_year|min_date|max_date|
+--------------+--------+--------+--------+--------+
|     australia|    2024|    2025|  202404|  202512|
|        brazil|    2006|    2026|  200601|  202602|
|        canada|    2006|    2026|  200601|  202602|
|         china|    2006|    2026|  200601|  202601|
|        france|    2006|    2026|  200601|  202602|
|       germany|    2006|    2026|  200601|  202602|
|         india|    2006|    2025|  200601|  202512|
|         italy|    2006|    2026|  200601|  202602|
|         japan|    2006|    2026|  200601|  202602|
|        mexico|    2006|    2026|  200601|  202602|
|        russia|    2006|    2026|  200601|  202601|
|  south_africa|    2006|    2026|  200601|  202602|
|   south_korea|    2006|    2026|  200601|  202602|
|       turkiye|    2006|    2026|  200601|  202602|
|united_kingdom|    2006|    2026|  200601|  202602|
| united_states|    2006|    2026|  200601|  2

##### Australian CPI fix

In [22]:
# Import dataset for Australian CPI
import_path = 'C:/Users/bravo/OneDrive/bravo_projects/MLProject/StraitofHormuzResearch/import_datasets/australiancpi.csv'
australiancpi_df = spark.read.csv(import_path, header=True, inferSchema=True)

# extract date
australiancpi_df = (
    australiancpi_df
    .withColumns({
        'country': F.lit('australia'),
        'year': F.substring('date', 5, 4).cast('int'),
        'month': F.substring('date', 1, 3)
    })
    .withColumn('month',  # cpi is reported quarterly.
                F.when(F.col('month') == 'Mar', 3)
                .when(F.col('month') == 'Jun', 6)
                .when(F.col('month') == 'Sep', 9)
                .when(F.col('month') == 'Dec', 12)
                .otherwise(None).cast('int')
                )
    .withColumnRenamed('cpi_index', 'australia_cpi')
    .drop('date')
)

# Join the Australian CPI data with the existing joined_df3
joined_df3 = joined_df3.join(australiancpi_df, on=['country', 'year', 'month'], how='left')

# Coalesce the cpi columns to fill in australian CPI values where available.
joined_df3 = (
    joined_df3
    .withColumn('cpi',
                F.when(F.col('country').contains('australia') & (F.col('year') == 2006) & (F.col('month').isin(3, 4, 5))
                       , F.lit(84.5))  # Correction for first cpi values in 2006 for australia, as the dataset starts from April 2006.
                .when(F.col('country').contains('australia') & F.col('cpi').isNull(), F.col('australia_cpi'))
                .otherwise(F.col('cpi'))
    ).drop('australia_cpi')
    # FFill missing cpi values for australia
    .withColumn('cpi',
                F.when(F.col('country').contains('australia'),
                       F.last(F.col('cpi'), ignorenulls=True).over(W.partitionBy('country').orderBy('year', 'month'))
                       ).otherwise(F.col('cpi'))
    )
)

In [23]:
joined_df3.filter('country = "australia" and month IN (7) and year = 2006').show(5)

+---------+----+-----+--------+-------------+-------+------------------------+----------------------+------+----+
|  country|year|month|    date|interest_rate|fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|events| cpi|
+---------+----+-----+--------+-------------+-------+------------------------+----------------------+------+----+
|australia|2006|    7|20060731|         5.75| 0.7664|                   74.75|                 74.56|     0|85.9|
|australia|2006|    7|20060728|         5.75| 0.7663|                   73.95|                  73.3|     0|85.9|
|australia|2006|    7|20060727|         5.75| 0.7641|                   75.36|                  74.5|     0|85.9|
|australia|2006|    7|20060726|         5.75| 0.7594|                   73.76|                 73.82|     0|85.9|
|australia|2006|    7|20060725|         5.75| 0.7529|                   72.49|                 73.46|     0|85.9|
+---------+----+-----+--------+-------------+-------+------------------------+----------

#### FFill correction for rest of missing Values

In [24]:
joined_df3 = (
    joined_df3
    .withColumn('cpi', F.last(F.col('cpi'), ignorenulls=True).over(W.partitionBy('country').orderBy('year', 'month')))
)

In [25]:
## Check for missing values
missing_values = {col: joined_df3.filter(joined_df3[col].isNull()).count() for col in joined_df3.columns}
print("Missing values in joined_df3:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df3:
--- --- --- --- --- --- --- --- ---
country: 0
year: 0
month: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 0


### Joined and GPR dataframe

In [26]:
# joined and gpr
joined_df4 = joined_df3.join(gpr_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df4.count(), len(joined_df4.columns))}")
print("Columns in joined_df4: ", joined_df4.columns)


## Check for duplicates
duplicate_count = joined_df4.groupBy(joined_df4.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df4: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 11)
Columns in joined_df4:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events', 'cpi', 'gpr_index']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df4: 0


### Check for missing values
No missing values found

In [27]:
## Check for missing values
missing_values = {col: joined_df4.filter(joined_df4[col].isNull()).count() for col in joined_df4.columns}
print("Missing values in joined_df4:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df4:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 0
gpr_index: 0


### Joined and LSCI dataset

In [28]:
# joined and lsci
joined_df5 = joined_df4.join(lsci_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df5.count(), len(joined_df5.columns))}")
print("Columns in joined_df5: ", joined_df5.columns)


## Check for duplicates
duplicate_count = joined_df5.groupBy(joined_df5.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df5: {duplicate_count}")


Joined DataFrame (row, col) count: (60520, 12)
Columns in joined_df5:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events', 'cpi', 'gpr_index', 'lsci']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df5: 0


#### Check for Missing Values

In [29]:
## Check for missing values
missing_values = {col: joined_df5.filter(joined_df5[col].isNull()).count() for col in joined_df5.columns}
print("Missing values in joined_df5:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df5:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 0
gpr_index: 0
lsci: 34437


##### Missing Values Treatment
LSCI data was reported on a quarterly frequency until 2022, then monthly afterwards. F-Fill is used to fill missing values, making the assumption that the recorded index value applies to proceding months.

In [30]:
lsci_df.show(5)

+---------+----+-----+------+
|  country|year|month|  lsci|
+---------+----+-----+------+
|australia|2006|    2|139.19|
|australia|2006|    5|141.65|
|australia|2006|    8| 142.0|
|australia|2006|   11|140.77|
|australia|2007|    2|141.92|
+---------+----+-----+------+
only showing top 5 rows



In [31]:
# lsci correction for australia for the months of March and April 2006 to ensure that these months have the correct February 2006 lsci value.
lsci_df = (
    lsci_df
    .withColumn('month', 
                F.when((F.col('year')==2006) & (F.col('month')==2)                                      # Corrected by Bryan Bravo on 2024-04-04: Missing from `fred_df`
                       & F.col('country').isin('germany', 'italy', 'russia', 'turkiye'), F.lit(3))      # Discovered that these countries are missing april data for 2006
                .when((F.col('year')==2006) & (F.col('month')==2), F.lit(4))
                .otherwise(F.col('month')))
)

# joined and gpr
joined_df5 = joined_df4.join(lsci_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df5.count(), len(joined_df5.columns))}")
print("Columns in joined_df5: ", joined_df5.columns)

# F-Fill to handle missing values in joined_df5 lsci.
joined_df5 = (
    joined_df5
    .withColumn('lsci', F.last(F.col('lsci'), ignorenulls=True).over(W.partitionBy('country').orderBy('year', 'month')))
)

# Create a dictionary that extracts the country and lsci for the month of February 2006 
# to fill in missing lsci values for australia for the months of March and April 2006
lsci_dict = {row['country']: row['lsci'] for row in lsci_df.filter((F.col('year')==2006)&(F.col('month')==2)).collect()}


Joined DataFrame (row, col) count: (60520, 12)
Columns in joined_df5:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events', 'cpi', 'gpr_index', 'lsci']


In [32]:
## Check for missing values
missing_values = {col: joined_df5.filter(joined_df5[col].isNull()).count() for col in joined_df5.columns}
print("Missing values in joined_df5:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df5:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 0
gpr_index: 0
lsci: 0


**Discovered that there is an inconsistency with the data on 2006-04**

Some countries are missing data from this month, beginning from the `fred_df` data on the initial join.
- The remedy for this is to drop all data from and before 2006-04. This still leaves us with plenty of data from 2006-05 and beyond.

In [33]:
joined_df5 = (
    joined_df5.filter(F.col('date')>20060430)
)
joined_df5.orderBy('date').show(50)

+----+-----+--------------+--------+----------------+--------------------+------------------------+----------------------+------+-----------------+--------------------+------+
|year|month|       country|    date|   interest_rate|             fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|events|              cpi|           gpr_index|  lsci|
+----+-----+--------------+--------+----------------+--------------------+------------------------+----------------------+------+-----------------+--------------------+------+
|2006|    5|     australia|20060501|5.72826086956522|              0.7607|                   73.37|                 73.75|     0|             84.5| 0.06284602731466293|141.65|
|2006|    5|        brazil|20060501|           15.75|  0.4811161895597787|                   73.37|                 73.75|     0|          2579.81| 0.06883136183023453| 164.8|
|2006|    5|        canada|20060501|          4.0283|  0.8984725965858041|                   73.37|                 73.7

### Joined with wb dataset

In [34]:
# joined and wb
joined_df6 = joined_df5.join(wb_df, on=['year', 'month', 'country'], how='left')
print(f"Joined DataFrame (row, col) count: {(joined_df6.count(), len(joined_df6.columns))}")
print("Columns in joined_df6: ", joined_df6.columns)


## Check for duplicates
duplicate_count = joined_df6.groupBy(joined_df6.columns).count().filter("count > 1").count()
print(f"--- --- --- --- --- --- --- --- ---\nNumber of duplicate rows in joined_df6: {duplicate_count}")


Joined DataFrame (row, col) count: (60276, 14)
Columns in joined_df6:  ['year', 'month', 'country', 'date', 'interest_rate', 'fx_rate', 'brent_dollars_per_barrel', 'wti_dollars_per_barrel', 'events', 'cpi', 'gpr_index', 'lsci', 'fx_reserves', 'imports_good_service']
--- --- --- --- --- --- --- --- ---
Number of duplicate rows in joined_df6: 0


In [35]:
## Check for missing values
missing_values = {col: joined_df6.filter(joined_df6[col].isNull()).count() for col in joined_df6.columns}
print("Missing values in joined_df6:\n--- --- --- --- --- --- --- --- ---")
for col, count in missing_values.items():
    print(f"{col}: {count}")


Missing values in joined_df6:
--- --- --- --- --- --- --- --- ---
year: 0
month: 0
country: 0
date: 0
interest_rate: 0
fx_rate: 0
brent_dollars_per_barrel: 0
wti_dollars_per_barrel: 0
events: 0
cpi: 0
gpr_index: 0
lsci: 0
fx_reserves: 4680
imports_good_service: 4680


##### Missing data from 2024-06 -> present
The missing data from 2024-06 to present on the `fx_reserves` and `imports_goods_service` is likely due to the fact that the World Bank has not yet released the data for these months.

In [36]:
joined_df6.filter('fx_reserves is null').orderBy('date').show(5)

+----+-----+-------+--------+-------------+-------------------+------------------------+----------------------+------+-----+------------------+-------+-----------+--------------------+
|year|month|country|    date|interest_rate|            fx_rate|brent_dollars_per_barrel|wti_dollars_per_barrel|events|  cpi|         gpr_index|   lsci|fx_reserves|imports_good_service|
+----+-----+-------+--------+-------------+-------------------+------------------------+----------------------+------+-----+------------------+-------+-----------+--------------------+
|2024|    6|  china|20240603|          2.9|0.13808340237503453|                   76.45|                 75.26|     0|102.9|0.8939893245697021|1210.86|       NULL|                NULL|
|2024|    6|  china|20240604|          2.9|0.13813490254582625|                   75.33|                 74.27|     0|102.9|0.8939893245697021|1210.86|       NULL|                NULL|
|2024|    6|  china|20240605|          2.9| 0.1379786133149362|            

In [37]:
joined_df6.filter(F.col('fx_reserves').isNull()).groupBy('country').count().orderBy('country').show()

+--------------+-----+
|       country|count|
+--------------+-----+
|     australia|  393|
|        brazil|  415|
|        canada|  393|
|         china|  269|
|        france|  393|
|         india|  415|
|         japan|  393|
|        mexico|  393|
|  south_africa|  415|
|   south_korea|  415|
|united_kingdom|  393|
| united_states|  393|
+--------------+-----+



#### Missing Data Treament - ML Imputation with Time Series Forecasting
Reasons for selecting ML imputation to fill missing data
- The missing window if from 2023-08 -> present, large temporal gap.
- Reserve Assets across major economies move together
- Other features can correlate with each other helping generate a more accurate model
- For an LSTM analysis we need continuity. (Future project idea)
- Avoiding unrealistic flatlines from F-Fill and produce realistic month-to-month variation

## ML imputation will be handled on the next notebook.

In [38]:
ml_df = (
    joined_df6.groupBy('year', 'month')
    .pivot('country')
    .agg(F.first('fx_reserves'))
    .orderBy('year', 'month')
)
ml_df.show(5)

+----+-----+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+
|year|month|       australia|          brazil|          canada|           china|          france|         germany|           india|           italy|           japan|          mexico|          russia|    south_africa|     south_korea|         turkiye|  united_kingdom|   united_states|
+----+-----+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+
|2006|    5| 49615.006092537|62675.5280265676|35613.0054447953|927527.635605136| 33534.487131222|43159.5673215149| 156844.64224677|24586.98948644

In [39]:
ml_df.filter('turkiye is not null').orderBy(F.desc('year'), F.desc('month')).show(1)

+----+-----+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+----------------+----------------+----------------+----------------+---------------+
|year|month|       australia|          brazil|          canada|           china|          france|         germany|           india|           italy|           japan|          mexico|         russia|    south_africa|     south_korea|         turkiye|  united_kingdom|  united_states|
+----+-----+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+----------------+----------------+----------------+----------------+---------------+
|2023|    8|54101.0640951812|336086.240427619|114173.662527447|3222723.05952683|77074.5238703132|99389.0042204953|553212.330477409|82872.1192301243|119

# Save Dataset

In [40]:
joined_df6.repartition(10).cache().count()  # Force cache to materialize and optimize for subsequent operations.

joined_df6.toPandas().to_csv(f"{out_path}/joined_input.csv", index=False)

spark.stop()